In [15]:
"""
Q2.1 从附件2 推导每台过滤器的"当前固定维护规律"
  - T_M_i: 历史中维护平均间隔（天）
  - T_L_i: 历史大维护平均间隔；若 <2 次大维护，标记为 inf
  - 也计算中位数作为鲁棒估计
输出:
  tables/fixed_maintenance_rule.csv
"""
import pandas as pd
import numpy as np
from pathlib import Path


In [16]:
PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT / "Q2输出"
mnt = pd.read_csv(PROJECT_ROOT / "Q1输出/data/maintenance_events.csv", parse_dates=["d"])
print(f"Loaded {len(mnt)} maintenance events")


Loaded 127 maintenance events


In [21]:

rows = []
for i in sorted(mnt["i"].unique()):
    sub = mnt[mnt["i"] == i].sort_values("d")
    m_dates = sub[sub["q"] == "m"]["d"].tolist()
    l_dates = sub[sub["q"] == "l"]["d"].tolist()

    # 中维护间隔
    if len(m_dates) >= 2:
        m_int = np.diff([d.toordinal() for d in m_dates])
        T_M_mean = np.mean(m_int)
        T_M_med = np.median(m_int)
    else:
        T_M_mean = T_M_med = np.nan

    # 大维护间隔
    if len(l_dates) >= 2:
        l_int = np.diff([d.toordinal() for d in l_dates])
        T_L_mean = np.mean(l_int)
        T_L_med = np.median(l_int)
    elif len(l_dates) == 1:
        # 只观测到 1 次，无法推间隔
        T_L_mean = T_L_med = np.nan
    else:
        # 0 次：未来也不做大维护
        T_L_mean = T_L_med = np.inf

    rows.append(dict(i=i, n_m=len(m_dates), n_l=len(l_dates),
                     T_M_mean=T_M_mean, T_M_med=T_M_med,
                     T_L_mean=T_L_mean, T_L_med=T_L_med,
                     last_m=m_dates[-1] if m_dates else pd.NaT,
                     last_l=l_dates[-1] if l_dates else pd.NaT))

rule = pd.DataFrame(rows)
print("\nPer-filter empirical maintenance cadence:")
print(rule.to_string(index=False))



Per-filter empirical maintenance cadence:
 i  n_m  n_l  T_M_mean  T_M_med  T_L_mean  T_L_med     last_m     last_l
 1   10    2 73.111111     62.0     347.0    347.0 2026-03-07 2025-11-14
 2    9    3 80.750000     56.5      57.5     57.5 2026-03-12 2026-01-29
 3   11    2 67.600000     55.5     138.0    138.0 2026-04-05 2025-11-17
 4   13    0 54.416667     57.0       inf      inf 2026-03-12        NaT
 5   12    1 62.181818     60.0       NaN      NaN 2026-04-03 2024-09-21
 6   10    2 73.000000     71.0     105.0    105.0 2026-03-25 2025-02-19
 7   11    2 67.600000     62.0     298.0    298.0 2026-03-25 2026-02-08
 8   13    0 55.583333     54.5       inf      inf 2026-03-31        NaT
 9   11    2 63.400000     62.5     321.0    321.0 2026-02-23 2026-04-01
10   10    3 74.444444     74.0     262.5    262.5 2026-03-26 2026-01-01


In [20]:

# 对只有 1 次大维护的 A5，用其余台的中位 T_L 填充
valid_TL = rule.loc[rule["T_L_mean"].notna() & np.isfinite(rule["T_L_mean"]),
                    "T_L_mean"]
T_L_fallback = np.median(valid_TL) if len(valid_TL) > 0 else 365.0
print(f"\nFallback T_L (median of valid): {T_L_fallback:.1f} days")

rule["T_M_use"] = rule["T_M_med"]
rule["T_L_use"] = rule["T_L_med"].copy()
mask_nan = rule["T_L_use"].isna()
rule.loc[mask_nan, "T_L_use"] = T_L_fallback
# 对 0 大维护的保持 inf
rule.loc[np.isinf(rule["T_L_mean"]), "T_L_use"] = np.inf

print("\nFinal 固定维护规律 (T_M_use, T_L_use)：")
print(rule[["i", "n_m", "n_l", "T_M_use", "T_L_use"]].to_string(index=False))

rule.to_csv(ROOT / "tables/fixed_maintenance_rule.csv", index=False)
print(f"\nSaved: {ROOT/'tables/fixed_maintenance_rule.csv'}")



Fallback T_L (median of valid): 262.5 days

Final 固定维护规律 (T_M_use, T_L_use)：
 i  n_m  n_l  T_M_use  T_L_use
 1   10    2     62.0    347.0
 2    9    3     56.5     57.5
 3   11    2     55.5    138.0
 4   13    0     57.0      inf
 5   12    1     60.0    262.5
 6   10    2     71.0    105.0
 7   11    2     62.0    298.0
 8   13    0     54.5      inf
 9   11    2     62.5    321.0
10   10    3     74.0    262.5

Saved: d:\Users\32174\Desktop\2026_math_modeling_competition\Q2输出\tables\fixed_maintenance_rule.csv


In [33]:
"""
Q2.2 扩展回归（加入 γ_m·C_m + γ_l·C_l）+ 训练/验证切分
  训练：2024-04-03 至 2025-09-30
  验证：2025-10-01 至 2026-01-19（保留 2026-01-20 起的缺测段）
  比较: base (无 γ) vs extended (含 γ)  →  验证集 RMSE
输出:
  data/Q2_design_with_C.csv
  tables/Q2_regression_compare.csv
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT / "Q2输出"
df = pd.read_csv(PROJECT_ROOT / "Q1输出/data/daily_with_vars.csv", parse_dates=["d"])
mnt = pd.read_csv(PROJECT_ROOT / "Q1输出/data/maintenance_events.csv", parse_dates=["d"])

# 为每 (i, d) 添加 C_m, C_l (累计维护次数)
def add_cum(df, mnt):
    out = []
    for i, sub in df.groupby("i"):
        sub = sub.sort_values("d").reset_index(drop=True).copy()
        ev = mnt[mnt["i"] == i].sort_values("d")
        m_dates = ev[ev["q"] == "m"]["d"].values
        l_dates = ev[ev["q"] == "l"]["d"].values
        d_arr = sub["d"].values
        sub["C_m"] = np.array([(m_dates <= d).sum() for d in d_arr])
        sub["C_l"] = np.array([(l_dates <= d).sum() for d in d_arr])
        out.append(sub)
    return pd.concat(out, ignore_index=True)

df = add_cum(df, mnt)
print(f"Added C_m (range {df['C_m'].min()}..{df['C_m'].max()}), "
      f"C_l (range {df['C_l'].min()}..{df['C_l'].max()})")
df.to_csv(ROOT / "data/Q2_design_with_C.csv", index=False)

# 只保留 y 非空
df = df.dropna(subset=["y"]).copy()
df["A_m_f"] = df["A_m"].fillna(0)
df["A_l_f"] = df["A_l"].fillna(0)
df["has_m"] = df["A_m"].notna().astype(int)
df["has_l"] = df["A_l"].notna().astype(int)

# 训练/验证切分
train_end = pd.Timestamp("2025-09-30")
val_start = pd.Timestamp("2025-10-01")
val_end   = pd.Timestamp("2026-01-19")

df_tr = df[df["d"] <= train_end].copy()
df_va = df[(df["d"] >= val_start) & (df["d"] <= val_end)].copy()
print(f"Train: {len(df_tr)} rows  ({df_tr['d'].min().date()} .. {df_tr['d'].max().date()})")
print(f"Valid: {len(df_va)} rows  ({df_va['d'].min().date()} .. {df_va['d'].max().date()})")

filters = sorted(df["i"].unique())

def design(df, include_gamma=False):
    parts = [np.ones(len(df))]; names = ["const"]
    # 过滤器 FE
    for i in filters[1:]:
        parts.append((df["i"] == i).astype(float).values); names.append(f"I(i={i})")
    # β_i·t
    for i in filters:
        parts.append(((df["i"] == i).astype(float) * df["t"]).values); names.append(f"t_i{i}")
    # 季节
    parts.append(df["sin1"].values); names.append("sin1")
    parts.append(df["cos1"].values); names.append("cos1")
    # 维护短期 & 漂移
    for col in ["H_m7", "H_l7", "A_m_f", "A_l_f", "has_m", "has_l"]:
        parts.append(df[col].values.astype(float)); names.append(col)
    if include_gamma:
        parts.append(df["C_m"].values.astype(float)); names.append("C_m")
        parts.append(df["C_l"].values.astype(float)); names.append("C_l")
    return np.column_stack(parts), names

def ols_fit(X, y):
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    n, k = X.shape
    sigma2 = (resid**2).sum() / (n - k)
    try:
        var = sigma2 * np.linalg.inv(X.T @ X)
        se = np.sqrt(np.diag(var))
    except Exception:
        se = np.full(k, np.nan)
    tstat = beta / np.where(se == 0, np.nan, se)
    r2 = 1 - (resid**2).sum() / ((y - y.mean())**2).sum()
    rmse = np.sqrt((resid**2).mean())
    return beta, se, tstat, r2, rmse

def eval_rmse(beta, X, y):
    r = y - X @ beta
    return np.sqrt((r**2).mean())

results = {}
for tag, inc in [("base", False), ("extended", True)]:
    Xtr, names = design(df_tr, include_gamma=inc)
    ytr = df_tr["y"].values
    beta, se, t, r2_tr, rmse_tr = ols_fit(Xtr, ytr)
    # 验证
    Xva, _ = design(df_va, include_gamma=inc)
    yva = df_va["y"].values
    rmse_va = eval_rmse(beta, Xva, yva)
    results[tag] = dict(beta=beta, names=names, r2_tr=r2_tr, rmse_tr=rmse_tr, rmse_va=rmse_va)
    print(f"\n{tag.upper()}:")
    print(f"  train R² = {r2_tr:.4f}, train RMSE = {rmse_tr:.3f}")
    print(f"  valid RMSE = {rmse_va:.3f}")
    if inc:
        j_m = names.index("C_m"); j_l = names.index("C_l")
        print(f"  γ_m = {beta[j_m]:+.3f} (SE={se[j_m]:.3f}, t={t[j_m]:+.2f})")
        print(f"  γ_l = {beta[j_l]:+.3f} (SE={se[j_l]:.3f}, t={t[j_l]:+.2f})")

# 汇总输出
rows = []
for tag in ["base", "extended"]:
    r = results[tag]
    rows.append(dict(model=tag,
                     train_R2=r["r2_tr"], train_RMSE=r["rmse_tr"],
                     valid_RMSE=r["rmse_va"]))
cmp = pd.DataFrame(rows).round(4)
print("\n对比：")
cmp.to_csv(ROOT / "tables/Q2_regression_compare.csv", index=False)


# 保存 extended 的完整系数
ex = results["extended"]
tab = pd.DataFrame({"var": ex["names"], "beta": ex["beta"]}).round(5)
tab.to_csv(ROOT / "tables/Q2_extended_coeffs.csv", index=False)

print(f"\nSaved: Q2_regression_compare.csv,  Q2_extended_coeffs.csv")


Added C_m (range 0..13), C_l (range 0..3)
Train: 4038 rows  (2024-04-03 .. 2025-09-30)
Valid: 1008 rows  (2025-10-01 .. 2026-01-19)

BASE:
  train R² = 0.8385, train RMSE = 7.413
  valid RMSE = 12.964

EXTENDED:
  train R² = 0.8516, train RMSE = 7.107
  valid RMSE = 14.168
  γ_m = +9.879 (SE=0.620, t=+15.94)
  γ_l = +22.117 (SE=1.390, t=+15.92)

对比：

Saved: Q2_regression_compare.csv,  Q2_extended_coeffs.csv


In [38]:
"""
Q2.2b 时序交叉验证选 (γ_m, γ_l)
  - 固定 (γ_m, γ_l) 网格值
  - 在训练集上最小化 Σ(y - γ_m*C_m - γ_l*C_l - Xβ)^2 对 β
  - 在验证集评估 RMSE
  - 选验证 RMSE 最小的 (γ_m, γ_l)
"""
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
df = pd.read_csv(ROOT / "Q2输出/data/Q2_design_with_C.csv", parse_dates=["d"])
df = df.dropna(subset=["y"]).copy()
df["A_m_f"] = df["A_m"].fillna(0)
df["A_l_f"] = df["A_l"].fillna(0)
df["has_m"] = df["A_m"].notna().astype(int)
df["has_l"] = df["A_l"].notna().astype(int)

train_end = pd.Timestamp("2025-09-30")
val_start = pd.Timestamp("2025-10-01")
val_end   = pd.Timestamp("2026-01-19")
df_tr = df[df["d"] <= train_end].copy()
df_va = df[(df["d"] >= val_start) & (df["d"] <= val_end)].copy()

filters = sorted(df["i"].unique())

def design_base(df):
    parts = [np.ones(len(df))]; names = ["const"]
    for i in filters[1:]:
        parts.append((df["i"] == i).astype(float).values); names.append(f"I(i={i})")
    for i in filters:
        parts.append(((df["i"] == i).astype(float) * df["t"]).values); names.append(f"t_i{i}")
    parts.append(df["sin1"].values); names.append("sin1")
    parts.append(df["cos1"].values); names.append("cos1")
    for col in ["H_m7", "H_l7", "A_m_f", "A_l_f", "has_m", "has_l"]:
        parts.append(df[col].values.astype(float)); names.append(col)
    return np.column_stack(parts), names

Xtr, names = design_base(df_tr)
Xva, _     = design_base(df_va)
ytr        = df_tr["y"].values
yva        = df_va["y"].values
Cm_tr, Cl_tr = df_tr["C_m"].values, df_tr["C_l"].values
Cm_va, Cl_va = df_va["C_m"].values, df_va["C_l"].values

# 网格：预期 γ<0 (负损伤)，但也允许 0 和小正值做对比
grid_m = np.array([-3, -2, -1, -0.5, -0.25, 0, 0.25])
grid_l = np.array([-10, -5, -2, -1, 0, 1, 2])

rows = []
best = (None, None, np.inf)
for gm in grid_m:
    for gl in grid_l:
        # 目标 y_adj = y - γ·C
        y_adj_tr = ytr - gm * Cm_tr - gl * Cl_tr
        beta, *_ = np.linalg.lstsq(Xtr, y_adj_tr, rcond=None)
        # 验证集预测：ŷ = X·β + γ·C
        yhat_va = Xva @ beta + gm * Cm_va + gl * Cl_va
        rmse_va = np.sqrt(((yva - yhat_va)**2).mean())
        # 训练集也记
        yhat_tr = Xtr @ beta + gm * Cm_tr + gl * Cl_tr
        rmse_tr = np.sqrt(((ytr - yhat_tr)**2).mean())
        rows.append(dict(gamma_m=gm, gamma_l=gl,
                         rmse_train=rmse_tr, rmse_valid=rmse_va))
        if rmse_va < best[2]:
            best = (gm, gl, rmse_va)

tab = pd.DataFrame(rows).round(4)
print("CV grid results:")
print(tab.pivot_table(index="gamma_m", columns="gamma_l", values="rmse_valid").round(3))
print(f"\nBest: γ_m = {best[0]}, γ_l = {best[1]}, valid RMSE = {best[2]:.3f}")

tab.to_csv(ROOT / "Q2输出/tables/gamma_cv_grid.csv", index=False)

# 使用最佳 γ 重新拟合，作为最终模型
gm, gl, _ = best
y_adj_tr = ytr - gm * Cm_tr - gl * Cl_tr
beta_final, *_ = np.linalg.lstsq(Xtr, y_adj_tr, rcond=None)
yhat_tr = Xtr @ beta_final + gm * Cm_tr + gl * Cl_tr
yhat_va = Xva @ beta_final + gm * Cm_va + gl * Cl_va
r2_tr = 1 - ((ytr-yhat_tr)**2).sum() / ((ytr-ytr.mean())**2).sum()
rmse_tr = np.sqrt(((ytr-yhat_tr)**2).mean())
rmse_va = np.sqrt(((yva-yhat_va)**2).mean())
print(f"\nFinal model:  train R²={r2_tr:.4f}, train RMSE={rmse_tr:.3f}, valid RMSE={rmse_va:.3f}")

# 保存
pd.DataFrame({"var": names, "beta": beta_final}).to_csv(
    ROOT / "Q2输出/tables/Q2_final_coeffs.csv", index=False)
pd.DataFrame([dict(gamma_m=gm, gamma_l=gl,
                   train_R2=r2_tr, train_RMSE=rmse_tr, valid_RMSE=rmse_va)]).to_csv(
    ROOT / "Q2输出/tables/Q2_final_summary.csv", index=False)
print(f"\nSaved: Q2_final_coeffs.csv, Q2_final_summary.csv, gamma_cv_grid.csv")


CV grid results:
gamma_l     -10     -5      -2      -1       0       1       2 
gamma_m                                                        
-3.00    13.727  13.221  13.053  13.021  13.000  12.992  12.996
-2.00    13.768  13.232  13.044  13.005  12.978  12.963  12.961
-1.00    13.819  13.252  13.045  13.000  12.966  12.944  12.935
-0.50    13.848  13.266  13.049  13.000  12.964  12.939  12.926
-0.25    13.863  13.274  13.052  13.002  12.963  12.937  12.922
 0.00    13.879  13.282  13.056  13.004  12.964  12.935  12.919
 0.25    13.895  13.291  13.060  13.006  12.964  12.935  12.917

Best: γ_m = 0.25, γ_l = 2, valid RMSE = 12.917

Final model:  train R²=0.8400, train RMSE=7.380, valid RMSE=12.917

Saved: Q2_final_coeffs.csv, Q2_final_summary.csv, gamma_cv_grid.csv


In [36]:
"""
Q2.3 分段线性衰减 vs 分段指数衰减比较
  线性: y_{i,d} = X·β + ε
  指数: log(y_{i,d}) = X·β' + ε'   →  y = exp(X·β')
  同一设计矩阵，比较训练 RMSE、验证 RMSE、残差结构
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT / "Q2输出"
df = pd.read_csv(ROOT / "data/Q2_design_with_C.csv", parse_dates=["d"])
df = df.dropna(subset=["y"]).copy()
df["A_m_f"] = df["A_m"].fillna(0)
df["A_l_f"] = df["A_l"].fillna(0)
df["has_m"] = df["A_m"].notna().astype(int)
df["has_l"] = df["A_l"].notna().astype(int)

train_end = pd.Timestamp("2025-09-30")
val_start = pd.Timestamp("2025-10-01")
val_end   = pd.Timestamp("2026-01-19")
df_tr = df[df["d"] <= train_end].copy()
df_va = df[(df["d"] >= val_start) & (df["d"] <= val_end)].copy()

filters = sorted(df["i"].unique())

def design(df):
    parts = [np.ones(len(df))]; names = ["const"]
    for i in filters[1:]:
        parts.append((df["i"] == i).astype(float).values); names.append(f"I(i={i})")
    for i in filters:
        parts.append(((df["i"] == i).astype(float) * df["t"]).values); names.append(f"t_i{i}")
    parts.append(df["sin1"].values); names.append("sin1")
    parts.append(df["cos1"].values); names.append("cos1")
    for col in ["H_m7", "H_l7", "A_m_f", "A_l_f", "has_m", "has_l"]:
        parts.append(df[col].values.astype(float)); names.append(col)
    return np.column_stack(parts), names

Xtr, names = design(df_tr)
Xva, _     = design(df_va)
ytr = df_tr["y"].values
yva = df_va["y"].values

def fit_eval(y_fit_tr, y_fit_va, transform="none"):
    beta, *_ = np.linalg.lstsq(Xtr, y_fit_tr, rcond=None)
    yhat_tr_raw = Xtr @ beta
    yhat_va_raw = Xva @ beta
    if transform == "log":
        yhat_tr = np.exp(yhat_tr_raw)
        yhat_va = np.exp(yhat_va_raw)
    else:
        yhat_tr = yhat_tr_raw
        yhat_va = yhat_va_raw
    r2_tr = 1 - ((ytr - yhat_tr)**2).sum() / ((ytr - ytr.mean())**2).sum()
    rmse_tr = np.sqrt(((ytr - yhat_tr)**2).mean())
    rmse_va = np.sqrt(((yva - yhat_va)**2).mean())
    mae_va = np.abs(yva - yhat_va).mean()
    return dict(beta=beta, r2_tr=r2_tr, rmse_tr=rmse_tr,
                rmse_va=rmse_va, mae_va=mae_va,
                yhat_tr=yhat_tr, yhat_va=yhat_va)

# 线性: 直接拟合 y
print("=== 线性模型 ===")
lin = fit_eval(ytr, yva, transform="none")
print(f"train R² = {lin['r2_tr']:.4f}, train RMSE = {lin['rmse_tr']:.3f}, "
      f"valid RMSE = {lin['rmse_va']:.3f}, valid MAE = {lin['mae_va']:.3f}")

# 指数: 拟合 log(y)
print("\n=== 指数模型 ===")
exp_model = fit_eval(np.log(ytr), np.log(yva), transform="log")
print(f"train R² = {exp_model['r2_tr']:.4f}, train RMSE = {exp_model['rmse_tr']:.3f}, "
      f"valid RMSE = {exp_model['rmse_va']:.3f}, valid MAE = {exp_model['mae_va']:.3f}")

# 分台比较：每台的 train RMSE / valid RMSE
df_tr["yhat_lin"] = lin["yhat_tr"]
df_tr["yhat_exp"] = exp_model["yhat_tr"]
df_va["yhat_lin"] = lin["yhat_va"]
df_va["yhat_exp"] = exp_model["yhat_va"]

rows = []
for i in filters:
    tr_i = df_tr[df_tr["i"] == i]
    va_i = df_va[df_va["i"] == i]
    def rmse(s1, s2): return np.sqrt(((s1 - s2)**2).mean())
    rows.append(dict(i=i,
                     n_tr=len(tr_i), n_va=len(va_i),
                     lin_tr=rmse(tr_i["y"], tr_i["yhat_lin"]),
                     exp_tr=rmse(tr_i["y"], tr_i["yhat_exp"]),
                     lin_va=rmse(va_i["y"], va_i["yhat_lin"]),
                     exp_va=rmse(va_i["y"], va_i["yhat_exp"])))
per_filter = pd.DataFrame(rows).round(3)
print("\n分台比较 (RMSE):")
print(per_filter.to_string(index=False))
per_filter.to_csv(ROOT / "tables/Q2_lin_vs_exp_per_filter.csv", index=False)

# 总体对比
overall = pd.DataFrame([
    dict(model="linear",      train_RMSE=lin["rmse_tr"],       valid_RMSE=lin["rmse_va"],       valid_MAE=lin["mae_va"]),
    dict(model="exponential", train_RMSE=exp_model["rmse_tr"], valid_RMSE=exp_model["rmse_va"], valid_MAE=exp_model["mae_va"]),
]).round(4)
print("\n总体：")
print(overall)
overall.to_csv(ROOT / "tables/Q2_lin_vs_exp_overall.csv", index=False)

# 保存最终选中的模型系数
winner = "linear" if lin["rmse_va"] <= exp_model["rmse_va"] else "exponential"
print(f"\n→ 择优: {winner}  (valid RMSE = "
      f"{lin['rmse_va'] if winner=='linear' else exp_model['rmse_va']:.3f})")
final_beta = lin["beta"] if winner == "linear" else exp_model["beta"]
pd.DataFrame({"var": names, "beta": final_beta}).to_csv(
    ROOT / "tables/Q2_winner_coeffs.csv", index=False)
with open(ROOT / "tables/Q2_winner.txt", "w") as f:
    f.write(winner)


=== 线性模型 ===
train R² = 0.8385, train RMSE = 7.413, valid RMSE = 12.964, valid MAE = 10.139

=== 指数模型 ===
train R² = 0.8153, train RMSE = 7.929, valid RMSE = 13.587, valid MAE = 10.230

分台比较 (RMSE):
 i  n_tr  n_va  lin_tr  exp_tr  lin_va  exp_va
 1   379    98   5.609   5.900  20.472  17.957
 2   391    99   5.275   5.154   6.173   3.959
 3   422   103   8.658   9.182  12.453  11.037
 4   416   102  10.217  11.800  24.264  28.833
 5   422   103   6.222   6.892  13.365  14.728
 6   413   101   8.133   8.000   6.299   5.524
 7   420   103   8.349   9.100   9.012   9.562
 8   402   101   5.121   5.452   6.172   6.418
 9   387   101   5.548   5.516  10.961   8.881
10   386    97   8.611   9.074   4.279   9.432

总体：
         model  train_RMSE  valid_RMSE  valid_MAE
0       linear      7.4126     12.9635    10.1389
1  exponential      7.9289     13.5872    10.2296

→ 择优: linear  (valid RMSE = 12.964)


In [39]:
"""
Q2.4 按固定维护规律向前外推 10 台透水率轨迹
  起点: 2026-01-20 (紧接观测末尾)
  外推期: 10 年 (3650 天)
  维护规则: T_M_i, T_L_i 来自 Q2.1
输出:
  data/future_trajectories.csv (i, d, y_sim)
  data/future_with_events.csv  (含未来维护事件)
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")

# 载入模型系数
coef = pd.read_csv(ROOT / "Q2输出/tables/Q2_winner_coeffs.csv")
beta_dict = dict(zip(coef["var"], coef["beta"]))
print(f"Loaded {len(beta_dict)} coefficients")

# 载入固定维护规律
rule = pd.read_csv(ROOT / "Q2输出/tables/fixed_maintenance_rule.csv")
rule["T_M_use"] = rule["T_M_use"].astype(float)
rule["T_L_use"] = rule["T_L_use"].astype(float)
print(rule[["i","T_M_use","T_L_use"]])

# 历史维护记录（用于 A_m, A_l, C_m, C_l 初始化）
mnt = pd.read_csv(ROOT / "Q1输出/data/maintenance_events.csv", parse_dates=["d"])

# 历史日表（用于获取 t 的锚点）
hist = pd.read_csv(ROOT / "Q1输出/data/daily_with_vars.csv", parse_dates=["d"])
anchor = hist["d"].min()  # 2024-04-03

# 外推配置
start_future = pd.Timestamp("2026-01-20")
H = 3650  # 10 年
future_days = pd.date_range(start_future, periods=H, freq="D")
T_year = 365.25

def simulate_filter(i, T_M, T_L, seed=0):
    """返回 DataFrame (d, y_sim, u_m, u_l, H_m7, H_l7, A_m, A_l)"""
    # 过去的维护
    past = mnt[mnt["i"] == i].sort_values("d")
    past_m = past[past["q"] == "m"]["d"].tolist()
    past_l = past[past["q"] == "l"]["d"].tolist()

    # 生成未来维护日程：从上次中维护日期 + T_M 开始
    last_m = past_m[-1] if past_m else anchor
    last_l = past_l[-1] if past_l else None

    future_m = []
    nxt = last_m + pd.Timedelta(days=int(round(T_M)))
    while nxt < future_days[-1]:
        future_m.append(nxt)
        nxt += pd.Timedelta(days=int(round(T_M)))

    future_l = []
    if np.isfinite(T_L):
        if last_l is None:
            # 0 大维护但 T_L 有限——不发生
            pass
        else:
            nxt = last_l + pd.Timedelta(days=int(round(T_L)))
            while nxt < future_days[-1]:
                future_l.append(nxt)
                nxt += pd.Timedelta(days=int(round(T_L)))

    all_m = past_m + future_m
    all_l = past_l + future_l

    # 为 future_days 计算每日协变量
    n = len(future_days)
    rec = pd.DataFrame(dict(d=future_days))
    rec["t"] = (rec["d"] - anchor).dt.days.astype(float)
    rec["sin1"] = np.sin(2*np.pi*rec["t"]/T_year)
    rec["cos1"] = np.cos(2*np.pi*rec["t"]/T_year)

    # u, H, A
    d_arr = rec["d"].values
    rec["u_m"] = [d in future_m for d in rec["d"]]
    rec["u_m"] = rec["u_m"].astype(int)
    rec["u_l"] = [d in future_l for d in rec["d"]]
    rec["u_l"] = rec["u_l"].astype(int)

    def H_window(dates, w=7):
        flag = np.zeros(n, dtype=int)
        for tau in dates:
            mask = (d_arr > np.datetime64(tau)) & \
                   (d_arr <= np.datetime64(tau + pd.Timedelta(days=w)))
            flag[mask] = 1
        return flag

    rec["H_m7"] = H_window(all_m)
    rec["H_l7"] = H_window(all_l)

    def days_since(dates):
        out = np.full(n, np.nan)
        last = None
        dates_sorted = sorted(dates)
        pt = 0
        for j, d in enumerate(rec["d"]):
            while pt < len(dates_sorted) and dates_sorted[pt] <= d:
                last = dates_sorted[pt]; pt += 1
            if last is not None:
                out[j] = (d - last).days
        return out

    rec["A_m"] = days_since(all_m)
    rec["A_l"] = days_since(all_l)
    rec["has_m"] = (~np.isnan(rec["A_m"])).astype(int)
    rec["has_l"] = (~np.isnan(rec["A_l"])).astype(int)
    rec["A_m_f"] = rec["A_m"].fillna(0)
    rec["A_l_f"] = rec["A_l"].fillna(0)

    # 计算 y_sim
    # 设计矩阵需要：const, I(i=2..10), t_ik (only t_i{i}=t, others=0), sin1, cos1, H_m7, H_l7, A_m_f, A_l_f, has_m, has_l
    y = np.full(n, beta_dict.get("const", 0.0))
    # 过滤器 FE
    if f"I(i={i})" in beta_dict:
        y += beta_dict[f"I(i={i})"]
    # β_i·t
    y += beta_dict[f"t_i{i}"] * rec["t"].values
    # 季节
    y += beta_dict["sin1"] * rec["sin1"].values
    y += beta_dict["cos1"] * rec["cos1"].values
    # 维护
    for col in ["H_m7", "H_l7", "A_m_f", "A_l_f", "has_m", "has_l"]:
        y += beta_dict[col] * rec[col].values

    rec["y_sim"] = y
    rec["i"] = i

    # 记录未来维护事件（用于画图）
    return rec[["i","d","t","y_sim","u_m","u_l","H_m7","H_l7","A_m","A_l",
                "has_m","has_l"]], future_m, future_l

all_trajs = []
future_events = []
for _, r in rule.iterrows():
    i = int(r["i"])
    T_M = r["T_M_use"]
    T_L = r["T_L_use"]
    traj, fm, fl = simulate_filter(i, T_M, T_L)
    all_trajs.append(traj)
    for d in fm: future_events.append(dict(i=i, d=d, q="m"))
    for d in fl: future_events.append(dict(i=i, d=d, q="l"))

fut = pd.concat(all_trajs, ignore_index=True)
fut.to_csv(ROOT / "Q2输出/data/future_trajectories.csv", index=False)

ev_df = pd.DataFrame(future_events).sort_values(["i","d"])
ev_df.to_csv(ROOT / "Q2输出/data/future_events.csv", index=False)

print(f"\nSimulated {len(fut)} future rows across {fut['i'].nunique()} filters")
print(f"Future maintenance events: {len(ev_df)}")
print(f"Time range: {fut['d'].min().date()} .. {fut['d'].max().date()}")

# 快速预览：每台第 1/12/60/120/365/1825 天的 y_sim
snap = fut.copy()
snap["days_from_start"] = (snap["d"] - start_future).dt.days
for k in [0, 30, 90, 180, 365, 730, 1095, 1460, 1825, 2555, 3285]:
    subset = snap[snap["days_from_start"] == k][["i","y_sim"]].set_index("i")["y_sim"].round(1)
    print(f"Day {k:4d}: ", subset.to_dict())


Loaded 28 coefficients
    i  T_M_use  T_L_use
0   1     62.0    347.0
1   2     56.5     57.5
2   3     55.5    138.0
3   4     57.0      inf
4   5     60.0    262.5
5   6     71.0    105.0
6   7     62.0    298.0
7   8     54.5      inf
8   9     62.5    321.0
9  10     74.0    262.5

Simulated 36500 future rows across 10 filters
Future maintenance events: 772
Time range: 2026-01-20 .. 2036-01-17
Day    0:  {1: 102.0, 2: 36.8, 3: 79.3, 4: 107.2, 5: 81.3, 6: 102.4, 7: 68.2, 8: 62.6, 9: 55.1, 10: 48.4}
Day   30:  {1: 91.6, 2: 31.3, 3: 69.4, 4: 112.4, 5: 72.1, 6: 108.2, 7: 78.5, 8: 63.0, 9: 45.5, 10: 60.8}
Day   90:  {1: 94.2, 2: 75.5, 3: 83.2, 4: 124.0, 5: 97.2, 6: 119.4, 7: 89.2, 8: 65.0, 9: 65.3, 10: 52.6}
Day  180:  {1: 112.4, 2: 92.9, 3: 81.9, 4: 152.8, 5: 98.5, 6: 133.1, 7: 92.1, 8: 80.3, 9: 78.9, 10: 49.5}
Day  365:  {1: 100.9, 2: 66.2, 3: 62.9, 4: 141.9, 5: 73.1, 6: 134.0, 7: 72.7, 8: 35.4, 9: 35.8, 10: 20.0}
Day  730:  {1: 107.5, 2: 69.1, 3: 33.6, 4: 169.1, 5: 55.4, 6: 170.3, 7

In [40]:
"""
Q2.5 寿命判定
   L_i = min { d : 滚动365日均值 ȳ_i(d) < 37 }
   另外检查 θ=4.33 判据：该时刻模拟再做一次大维护能否把 ȳ 拉回 ≥37+θ
输出:
   tables/life_prediction.csv
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
fut = pd.read_csv(ROOT / "Q2输出/data/future_trajectories.csv", parse_dates=["d"])

# 历史观测 + 未来外推 合并，以便滚动 365 日窗口在过渡期也能计算
hist = pd.read_csv(ROOT / "Q1输出/data/daily_with_vars.csv", parse_dates=["d"])
hist_slim = hist[["i","d","y"]].copy().rename(columns={"y":"y_obs"})
merged = hist_slim.merge(fut[["i","d","y_sim"]], on=["i","d"], how="outer")
# 用观测优先，未观测用模拟
merged["y_mix"] = merged["y_obs"].fillna(merged["y_sim"])
merged = merged.sort_values(["i","d"]).reset_index(drop=True)

# 滚动 365 天均值（每台独立）
merged = merged.sort_values(["i", "d"]).reset_index(drop=True)
merged["rolling_avg_365"] = (
    merged.groupby("i")["y_mix"]
    .transform(lambda s: s.rolling(window=365, min_periods=180).mean())
)
merged.to_csv(ROOT / "Q2输出/data/full_with_rolling.csv", index=False)

# 观测期末的滚动均值（2026-01-19 之前）
cutoff = pd.Timestamp("2026-01-19")
start_future = pd.Timestamp("2026-01-20")

# 寿命：未来段首次 rolling_avg < 37 的日期
rows = []
theta = 4.33
eta_l = 2.25
has_l_bump = 14.68   # from Q1 coeffs — first-time large maintenance adds ~14.68
# Approx improvement for another large maintenance: Δ^l ≈ η_l (short-term pulse)
# If first maintenance hasn't happened yet, additional jump is from has_l; else just η_l
# We'll use η_l + 7 days worth of -ρ_l = -0.21 (approx) = +2.04 net over 7 days
# This is simplified; just use η_l as the representative improvement.

for i in sorted(merged["i"].unique()):
    sub = merged[merged["i"] == i].copy()
    # 只在未来段中找寿命
    fut_sub = sub[sub["d"] >= start_future].reset_index(drop=True)
    below = fut_sub[fut_sub["rolling_avg_365"] < 37]
    if len(below) == 0:
        L_days = np.inf
        L_date = pd.NaT
        L_year = np.inf
        rolling_at_L = fut_sub["rolling_avg_365"].iloc[-1]
    else:
        L_date = below["d"].iloc[0]
        L_days = (L_date - start_future).days
        L_year = L_days / 365.25
        rolling_at_L = below["rolling_avg_365"].iloc[0]

    # 该时刻"再做一次大维护"的预测改善量（用 η_l 近似）
    delta_l_at_L = eta_l  # conservative lower bound — only the H_l7 pulse

    # 起始（2026-01-20）时的滚动均值
    rolling_start = fut_sub["rolling_avg_365"].iloc[0]

    rows.append(dict(
        i=i,
        L_days=L_days,
        L_year=L_year,
        L_date=L_date,
        rolling_avg_at_start=round(rolling_start, 2) if pd.notna(rolling_start) else None,
        rolling_avg_at_L=round(rolling_at_L, 2) if pd.notna(rolling_at_L) else None,
        delta_l_predicted=delta_l_at_L,
        theta=theta,
        retirement_triggered=(L_days != np.inf) and (delta_l_at_L < theta),
    ))

life = pd.DataFrame(rows)
print("10 台寿命预测结果：")
print(life.to_string(index=False))

life.to_csv(ROOT / "Q2输出/tables/life_prediction.csv", index=False)
print(f"\nSaved: Q2输出/tables/life_prediction.csv")


10 台寿命预测结果：
 i  L_days   L_year     L_date  rolling_avg_at_start  rolling_avg_at_L  delta_l_predicted  theta  retirement_triggered
 1     inf      inf        NaT                 90.83            136.89               2.25   4.33                 False
 2     inf      inf        NaT                 71.02             45.18               2.25   4.33                 False
 3  1191.0 3.260780 2029-04-25                 90.61             36.99               2.25   4.33                  True
 4     inf      inf        NaT                100.88            426.17               2.25   4.33                 False
 5  2320.0 6.351814 2032-05-28                 86.73             36.99               2.25   4.33                  True
 6     inf      inf        NaT                 85.61            439.47               2.25   4.33                 False
 7     inf      inf        NaT                 88.22             38.73               2.25   4.33                 False
 8   691.0 1.891855 2027-12-12      

In [41]:
"""
Q2.6 输出图表与总结
  fig1: 10 台过滤器的 y 轨迹（历史观测 + 未来预测）
  fig2: 10 台滚动 365 日均值曲线 + 37 阈值线
  fig3: 预测寿命条形图
  Q2_summary.md: 关键数字与解读
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")

fut = pd.read_csv(ROOT / "Q2输出/data/future_trajectories.csv", parse_dates=["d"])
hist = pd.read_csv(ROOT / "Q1输出/data/daily_with_vars.csv", parse_dates=["d"])
merged = pd.read_csv(ROOT / "Q2输出/data/full_with_rolling.csv", parse_dates=["d"])
life = pd.read_csv(ROOT / "Q2输出/tables/life_prediction.csv", parse_dates=["L_date"])
rule = pd.read_csv(ROOT / "Q2输出/tables/fixed_maintenance_rule.csv")

start_future = pd.Timestamp("2026-01-20")

# ---------- Fig 1: 10 台 y 轨迹 ----------
fig, axes = plt.subplots(5, 2, figsize=(16, 16), sharex=True)
for ax, i in zip(axes.ravel(), range(1, 11)):
    h = hist[hist["i"] == i]
    f = fut[fut["i"] == i]
    ax.plot(h["d"], h["y"], ".", ms=1, c="steelblue", alpha=0.5, label="observed")
    ax.plot(f["d"], f["y_sim"], "-", lw=0.6, c="crimson", alpha=0.7, label="simulated")
    ax.axhline(37, c="red", ls="--", lw=0.7)
    ax.axvline(start_future, c="k", ls=":", lw=0.6)
    r = life[life["i"] == i].iloc[0]
    if pd.notna(r["L_date"]):
        ax.axvline(r["L_date"], c="darkred", ls="-.", lw=0.9,
                   label=f"L={r['L_year']:.2f}y")
    ax.set_title(f"A{i}  T_M={rule[rule['i']==i]['T_M_use'].iloc[0]:.0f}d, "
                 f"T_L={rule[rule['i']==i]['T_L_use'].iloc[0]}d",
                 fontsize=9)
    ax.set_ylim(-50, 250)
    ax.legend(loc="upper right", fontsize=7)
    ax.grid(alpha=0.3)
fig.suptitle("Q2  permeability y: observed (blue) + simulated under fixed maintenance rule (red)",
             fontsize=13, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.975])
fig.savefig(ROOT / "Q2输出/figures/fig_q2_trajectories.png", dpi=110)
plt.close(fig)
print("Saved fig_q2_trajectories.png")

# ---------- Fig 2: 滚动 365 日均值 ----------
fig, axes = plt.subplots(5, 2, figsize=(16, 16), sharex=True, sharey=True)
for ax, i in zip(axes.ravel(), range(1, 11)):
    m = merged[merged["i"] == i].sort_values("d")
    ax.plot(m["d"], m["rolling_avg_365"], "-", lw=1, c="darkblue")
    ax.axhline(37, c="red", ls="--", lw=0.7, label="threshold 37")
    ax.axvline(start_future, c="k", ls=":", lw=0.6)
    r = life[life["i"] == i].iloc[0]
    if pd.notna(r["L_date"]):
        ax.axvline(r["L_date"], c="darkred", ls="-.", lw=0.9,
                   label=f"retires {r['L_date'].strftime('%Y-%m')}")
    ax.set_title(f"A{i}", fontsize=9)
    ax.set_ylim(0, 250)
    ax.legend(loc="upper right", fontsize=7)
    ax.grid(alpha=0.3)
fig.suptitle("Q2  Rolling 365-day mean permeability (threshold: y=37)", fontsize=13, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.975])
fig.savefig(ROOT / "Q2输出/figures/fig_q2_rolling_avg.png", dpi=110)
plt.close(fig)
print("Saved fig_q2_rolling_avg.png")

# ---------- Fig 3: 寿命条形图 ----------
fig, ax = plt.subplots(figsize=(9, 4.5))
life_plot = life.sort_values("L_year")
# 替换 inf 为 10+
cap = 10.5
life_plot["L_plot"] = life_plot["L_year"].replace(np.inf, cap)
colors = ["#d9534f" if y < 3 else "#f0ad4e" if y < 7 else "#5cb85c" for y in life_plot["L_plot"]]
bars = ax.barh([f"A{i}" for i in life_plot["i"]],
               life_plot["L_plot"], color=colors, edgecolor="k")
ax.axvline(1, c="k", ls="--", lw=0.5)
ax.axvline(5, c="k", ls="--", lw=0.5)
ax.set_xlabel("Predicted remaining life (years from 2026-01-20)")
ax.set_title("Q2  10-filter life prediction under fixed maintenance rule")
for bar, L, d in zip(bars, life_plot["L_year"], life_plot["L_date"]):
    txt = f">{cap:.0f}y" if np.isinf(L) else f"{L:.2f}y  ({d.strftime('%Y-%m')})"
    ax.text(min(L, cap), bar.get_y()+bar.get_height()/2, "  "+txt,
            va="center", fontsize=9)
ax.set_xlim(0, cap+2)
ax.grid(alpha=0.3, axis="x")
fig.tight_layout()
fig.savefig(ROOT / "Q2输出/figures/fig_q2_life_bar.png", dpi=130)
plt.close(fig)
print("Saved fig_q2_life_bar.png")


Saved fig_q2_trajectories.png
Saved fig_q2_rolling_avg.png
Saved fig_q2_life_bar.png
